# Washington Hikes — Data Ingestion

Scrapes trail information from multiple sources and loads chunked, embedded documents into a persistent local Chroma vector store.

**Sources:**
- **Wikipedia** — detailed articles on trails, wilderness areas, national parks, and national forests across Washington state
- **National Park Service** — main, plan-your-visit, and weather/seasons pages for Mt Rainier, Olympic, and North Cascades
- **Recreation.gov** — permit pages with seasonal access windows and quota info

> **Note:** Washington Trails Association (WTA) and US Forest Service (USFS) pages were excluded — WTA is behind an aggressive Cloudflare block, and USFS activity pages are JavaScript-rendered and return no content with a plain HTTP scraper.

**Required `.env` file:**
```
OPENAI_API_KEY=sk-...
```

In [1]:
import os
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langchain_community")

from dotenv import load_dotenv
load_dotenv()

os.environ.setdefault("USER_AGENT", "WAHikesRAGBot/1.0 (educational RAG project)")

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

/var/folders/hp/2b8j_s5x41794z8q1d1kqnyr0000gn/T/ipykernel_79372/3568154211.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


## 1. Define URLs

Three source types, all verified to load cleanly:

- **NPS** — main, plan-your-visit, and **weather pages** per park; weather pages have explicit seasonal guidance (e.g. "best weather mid-June to late-September")
- **Wikipedia** — articles on individual trails, wilderness areas, national parks, and national forests; richest content (2K–83K chars per page)
- **Recreation.gov** — permit pages include seasonal access windows and quota info (e.g. Enchantments lottery runs May–October)

> To expand coverage, add more Wikipedia article slugs or NPS park codes (`nps.gov/{park-code}/planyourvisit/weather.htm`).

In [2]:
# ----- National Park Service -----
nps_urls = [
    # Mt Rainier
    "https://www.nps.gov/mora/index.htm",
    "https://www.nps.gov/mora/planyourvisit/index.htm",
    "https://www.nps.gov/mora/planyourvisit/weather.htm",
    # Olympic
    "https://www.nps.gov/olym/index.htm",
    "https://www.nps.gov/olym/planyourvisit/index.htm",
    "https://www.nps.gov/olym/planyourvisit/weather.htm",
    # North Cascades
    "https://www.nps.gov/noca/index.htm",
    "https://www.nps.gov/noca/planyourvisit/hiking.htm",
    "https://www.nps.gov/noca/planyourvisit/weather.htm",
]

# ----- Wikipedia — trails, parks, wilderness areas -----
wikipedia_urls = [
    # Long-distance / iconic trails
    "https://en.wikipedia.org/wiki/Wonderland_Trail",
    "https://en.wikipedia.org/wiki/Pacific_Crest_Trail",
    "https://en.wikipedia.org/wiki/The_Enchantments",
    # Individual trails / peaks
    "https://en.wikipedia.org/wiki/Mount_Si",
    "https://en.wikipedia.org/wiki/Rattlesnake_Ledge",
    "https://en.wikipedia.org/wiki/North_Cascades_Highway",  # covers Maple Pass Loop area
    # Wilderness areas
    "https://en.wikipedia.org/wiki/Goat_Rocks_Wilderness",
    "https://en.wikipedia.org/wiki/Alpine_Lakes_Wilderness",
    "https://en.wikipedia.org/wiki/Glacier_Peak_Wilderness",
    "https://en.wikipedia.org/wiki/Pasayten_Wilderness",
    # National parks & forests
    "https://en.wikipedia.org/wiki/Mount_Rainier_National_Park",
    "https://en.wikipedia.org/wiki/Olympic_National_Park",
    "https://en.wikipedia.org/wiki/North_Cascades_National_Park",
    "https://en.wikipedia.org/wiki/Mount_Baker%E2%80%93Snoqualmie_National_Forest",
    "https://en.wikipedia.org/wiki/Okanogan%E2%80%93Wenatchee_National_Forest",
    "https://en.wikipedia.org/wiki/Olympic_National_Forest",
    # Regions
    "https://en.wikipedia.org/wiki/Olympic_Peninsula",
    "https://en.wikipedia.org/wiki/North_Cascades",
    "https://en.wikipedia.org/wiki/Issaquah_Alps",
]

# ----- Recreation.gov — permit & seasonal access info -----
recreation_urls = [
    "https://www.recreation.gov/permits/233273",  # Enchantments — permit windows, May–Oct season
]

all_urls = nps_urls + wikipedia_urls + recreation_urls
print(f"Total URLs: {len(all_urls)}")

Total URLs: 29


## 2. Scrape

Single `WebBaseLoader` pass across all 25 URLs. `continue_on_failure=True` skips any URL that errors rather than crashing the whole run.

In [3]:
loader = WebBaseLoader(
    web_paths=all_urls,
    bs_get_text_kwargs={"separator": " ", "strip": True},
    continue_on_failure=True,
)
loader.requests_per_second = 2

docs = loader.load()
print(f"Loaded {len(docs)} documents")

Loaded 29 documents


In [7]:
for doc in docs[:5]:
    print("Source :", doc.metadata.get("source"))
    print("Chars  :", len(doc.page_content))
    print(doc.page_content[2000:3000])
    print("=" * 70)

Source : https://www.nps.gov/mora/index.htm
Chars  : 3475
mbing Information and permits for exploring Mount Rainier's backcountry. Spring Hiking Safety (April-June) It's easy to underestimate the hazards of hiking at higher elevations still covered in snow. Follow these tips for safely hiking on snow. Hiking Safety No matter the length of your hike, being aware of risks can help make your hiking experience safe and enjoyable! Park Construction Park construction includes work to improve roads and visitor areas. Learn how this will impact your travel plans through the park. Accessibility at Mount Rainier Learn more about accessibility at Mount Rainier National Park. Mount Rainier Valor Memorial Ensuring that the individuals who made the ultimate sacrifice while saving the lives of others will never be forgotten. Last updated: May 5, 2026 Explore Life at the Top! Mount Rainier Virtual Tour Learn About Your Park Winter at Mount Rainier Glaciers Park footer Contact Info Mailing Address: 552

## 3. Clean & Split

In [8]:
# Drop pages that returned essentially no content (login walls, 404s, etc.)
MIN_CONTENT_LENGTH = 300
docs_filtered = [d for d in docs if len(d.page_content.strip()) >= MIN_CONTENT_LENGTH]

dropped = len(docs) - len(docs_filtered)
print(f"Kept {len(docs_filtered)} docs  |  dropped {dropped} near-empty pages")

Kept 29 docs  |  dropped 0 near-empty pages


In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

splits = splitter.split_documents(docs_filtered)
print(f"Total chunks: {len(splits)}")
print(f"\nSample chunk:\n{splits[0].page_content[:400]}")

Total chunks: 640

Sample chunk:
Mount Rainier National Park (U.S. National Park Service) An official website of the United States government Here's how you know Here's how you know Official websites use .gov A .gov website belongs to an official government
organization in the United States. Secure .gov websites use HTTPS A lock ( Lock Locked padlock icon ) or https:// means you've safely connected to
the .gov website. Share sens


## 4. Embed & Load into Chroma

In [10]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name="washington_hikes",
)

print(f"Stored {vectorstore._collection.count()} chunks in ./chroma_db")

Stored 640 chunks in ./chroma_db


## 5. Smoke Test

In [12]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

test_queries = [
    "easy hikes near Seattle with great views",
    "hardest hikes in Washington state",
    "hikes near Mount Rainier under 5 miles",
    "current trail conditions in North Cascades",
    "hikes suitable for kids on the Olympic Peninsula",
    "hikes with waterfalls in Washington",
]

for q in test_queries:
    print(f"\nQ: {q}")
    results = retriever.invoke(q)
    for r in results:
        src = r.metadata.get("source", "unknown")
        snippet = r.page_content[:500].replace("\n", " ")
        print(f"  [{src}]")
        print(f"  {snippet}...")


Q: easy hikes near Seattle with great views
  [https://en.wikipedia.org/wiki/Issaquah_Alps]
  D.; Hyndman, Donald W. (1984). Roadside Geology of Washington . Missoula: Mountain Press. ISBN 0-87842-160-2 . External links [ edit ] Issaquah Alps Trails Club Trail reviews of hikes in the Issaquah Alps at Hiking with my Brother 47°30′N 122°00′W ﻿ / ﻿ 47.500°N 122.000°W ﻿ / 47.500; -122.000 v t e Washington hills and ridges Eastern Washington Yakima Fold Belt Ahtanum Ridge Badger Mountain Beezley Hills Candy Mountain Columbia Hills Frenchman Hills Horse Heaven Hills ( Jump Off Joe ) Manastash ...
  [https://en.wikipedia.org/wiki/North_Cascades_National_Park]
  haze. [ 118 ] Climate change will impact the temperatures of high altitude lakes and streams, which in turn will have an effect on the fish that can thrive in these waters. Retreating glaciers reduce the amount of glacial ice melt available in warmer months that kept streams and lakes cold, even in late summer. [ 72 ] Attractions [ ed